In [ ]:
import pandas as pd
import plotly.graph_objects as go

def plot_prosperity_day(day_str, product='ASH_COATED_OSMIUM'):
    """
    Loads data for a specific day and plots the interactive candlestick chart
    with all 3 levels of order book depth without cluttering the view.
    """
    # 1. Load the data for the requested day
    df_prices = pd.read_csv(f'dataset/prices_round_1_day_{day_str}.csv', sep=';') 
    df_trades = pd.read_csv(f'dataset/trades_round_1_day_{day_str}.csv', sep=';')

    # Filter datasets for the chosen product
    p_df = df_prices[df_prices['product'] == product].copy()
    t_df = df_trades[df_trades['symbol'] == product].copy()

    # 2. Aggregate into 500-timestamp candles for the Mid-Price
    candle_size = 500
    p_df['candle_bin'] = (p_df['timestamp'] // candle_size) * candle_size

    ohlc = p_df.groupby('candle_bin').agg(
        Open=('mid_price', 'first'),
        High=('mid_price', 'max'),
        Low=('mid_price', 'min'),
        Close=('mid_price', 'last')
    ).reset_index()

    fig = go.Figure()

    # -------------------------------------------------------------
    # 3. ADD ORDER BOOK DEPTH (Plotted first to stay in background)
    # -------------------------------------------------------------
    
    # BIDS - Fading blue lines for depth (Opacity: 50% -> 25% -> 10%)
    bid_colors = ['rgba(50, 100, 250, 1)', 'rgba(50, 100, 250, 0.75)', 'rgba(50, 100, 250, 0.5)']
    for i in range(1, 4):
        col_name = f'bid_price_{i}'
        if col_name in p_df.columns:
            fig.add_trace(go.Scatter(
                x=p_df['timestamp'],
                y=p_df[col_name],
                mode='lines',
                line=dict(color=bid_colors[i-1], width=1 if i==1 else 0.5), # Thinner lines for L2 & L3
                name=f'Bid L{i}',
                hoverinfo='skip' 
            ))

    # ASKS - Fading red lines for depth (Opacity: 50% -> 25% -> 10%)
    ask_colors = ['rgba(250, 50, 50, 1)', 'rgba(250, 50, 50, 0.75)', 'rgba(250, 50, 50, 0.5)']
    for i in range(1, 4):
        col_name = f'ask_price_{i}'
        if col_name in p_df.columns:
            fig.add_trace(go.Scatter(
                x=p_df['timestamp'],
                y=p_df[col_name],
                mode='lines',
                line=dict(color=ask_colors[i-1], width=1 if i==1 else 0.5), # Thinner lines for L2 & L3
                name=f'Ask L{i}',
                hoverinfo='skip'
            ))

    # -------------------------------------------------------------
    # 4. ADD FOREGROUND ELEMENTS (Candlesticks & Trades)
    # -------------------------------------------------------------
    
    # Add Candlesticks (Mid Price)
    fig.add_trace(go.Candlestick(
        x=ohlc['candle_bin'],
        open=ohlc['Open'],
        high=ohlc['High'],
        low=ohlc['Low'],
        close=ohlc['Close'],
        name='Mid Price',
        increasing_line_color="#1cbd6f", # Teal
        decreasing_line_color="#b11411"  # Red
    ))

    # Add Trade Markers
    if not t_df.empty:
        fig.add_trace(go.Scatter(
            x=t_df['timestamp'],
            y=t_df['price'],
            mode='markers',
            marker=dict(color='#ffca28', size=6, symbol='circle'), # Yellow dots
            name='Trades'
        ))

    # 5. Format Layout for Interactivity and Dark Theme
    fig.update_layout(
        title=f'{product} - Day {day_str} ({candle_size}-timestamp candles) w/ Order Depth',
        xaxis_title='Timestamp',
        yaxis_title='Price (XIRECs)',
        template='plotly_dark', 
        xaxis_rangeslider_visible=True, # Adjustable range slider
        hovermode='x unified', 
        height=700
    )

    # Display the interactive chart in the browser
    fig.show(renderer="browser")

# --- RUN THE VISUALIZATIONS ---

# This will render the chart for Day -2
for prd in ["ASH_COATED_OSMIUM", "INTARIAN_PEPPER_ROOT"]:
    for i in ["-2", "-1", "0"]:
        plot_prosperity_day(i, prd)

In [1]:
import pandas as pd
import plotly.graph_objects as go

def plot_prosperity_day(day_str, product='ASH_COATED_OSMIUM'):
    """
    Loads data for a specific day and plots the interactive candlestick chart
    with all 3 levels of order book depth without cluttering the view.
    """
    # 1. Load the data for the requested day
    df_prices = pd.read_csv(f'dataset/prices_round_1_day_{day_str}.csv', sep=';') 
    df_trades = pd.read_csv(f'dataset/trades_round_1_day_{day_str}.csv', sep=';')

    # Filter datasets for the chosen product
    p_df = df_prices[df_prices['product'] == product].copy()
    t_df = df_trades[df_trades['symbol'] == product].copy()

    # --- FIX 1: Filter out invalid ticks where the order book clears out ---
    p_df = p_df[p_df['mid_price'] > 0]

    # 2. Aggregate into 500-timestamp candles for the Mid-Price
    candle_size = 500
    p_df['candle_bin'] = (p_df['timestamp'] // candle_size) * candle_size

    ohlc = p_df.groupby('candle_bin').agg(
        Open=('mid_price', 'first'),
        High=('mid_price', 'max'),
        Low=('mid_price', 'min'),
        Close=('mid_price', 'last')
    ).reset_index()

    fig = go.Figure()

    # -------------------------------------------------------------
    # 3. ADD ORDER BOOK DEPTH (Plotted first to stay in background)
    # -------------------------------------------------------------
    
    # BIDS - Fading blue lines for depth (Opacity: 50% -> 25% -> 10%)
    bid_colors = ['rgba(50, 100, 250, 1)', 'rgba(50, 100, 250, 0.75)', 'rgba(50, 100, 250, 0.5)']
    for i in range(1, 4):
        col_name = f'bid_price_{i}'
        if col_name in p_df.columns:
            fig.add_trace(go.Scatter(
                x=p_df['timestamp'],
                y=p_df[col_name],
                mode='lines',
                line=dict(color=bid_colors[i-1], width=1 if i==1 else 0.5), # Thinner lines for L2 & L3
                name=f'Bid L{i}',
                hoverinfo='skip' 
            ))

    # ASKS - Fading red lines for depth (Opacity: 50% -> 25% -> 10%)
    ask_colors = ['rgba(250, 50, 50, 1)', 'rgba(250, 50, 50, 0.75)', 'rgba(250, 50, 50, 0.5)']
    for i in range(1, 4):
        col_name = f'ask_price_{i}'
        if col_name in p_df.columns:
            fig.add_trace(go.Scatter(
                x=p_df['timestamp'],
                y=p_df[col_name],
                mode='lines',
                line=dict(color=ask_colors[i-1], width=1 if i==1 else 0.5), # Thinner lines for L2 & L3
                name=f'Ask L{i}',
                hoverinfo='skip'
            ))

    # -------------------------------------------------------------
    # 4. ADD FOREGROUND ELEMENTS (Candlesticks & Trades)
    # -------------------------------------------------------------
    
    # Add Candlesticks (Mid Price)
    fig.add_trace(go.Candlestick(
        x=ohlc['candle_bin'],
        open=ohlc['Open'],
        high=ohlc['High'],
        low=ohlc['Low'],
        close=ohlc['Close'],
        name='Mid Price',
        increasing_line_color="#1cbd6f", # Teal
        decreasing_line_color="#b11411"  # Red
    ))

    # Add Trade Markers
    if not t_df.empty:
        fig.add_trace(go.Scatter(
            x=t_df['timestamp'],
            y=t_df['price'],
            mode='markers',
            marker=dict(color='#ffca28', size=6, symbol='circle'), # Yellow dots
            name='Trades',
            # --- FIX 3: Add Volume info to tooltip using customdata ---
            customdata=t_df['quantity'],
            hovertemplate='Price: %{y}<br>Volume: %{customdata}<extra></extra>'
        ))

    # 5. Format Layout for Interactivity and Dark Theme
    fig.update_layout(
        title=f'{product} - Day {day_str} ({candle_size}-timestamp candles) w/ Order Depth',
        xaxis_title='Timestamp',
        yaxis_title='Price (XIRECs)',
        template='plotly_dark', 
        xaxis_rangeslider_visible=True, # Adjustable range slider
        # --- FIX 2: Allow Y-axis to be resizeable/zoomable ---
        yaxis=dict(fixedrange=False), 
        hovermode='x unified', 
        height=700
    )

    # Display the interactive chart in the browser
    fig.show(renderer="browser")

# --- RUN THE VISUALIZATIONS ---

# This will render the chart for Day -2
for prd in ["ASH_COATED_OSMIUM", "INTARIAN_PEPPER_ROOT"]:
    for i in ["-2", "-1", "0"]:
        plot_prosperity_day(i, prd)

In [3]:
import pandas as pd
import plotly.graph_objects as go

def plot_prosperity_day(day_str, product='ASH_COATED_OSMIUM'):
    """
    Loads data for a specific day and plots the interactive line chart
    with EMAs and all 3 levels of order book depth.
    """
    # 1. Load the data for the requested day
    df_prices = pd.read_csv(f'dataset/prices_round_2_day_{day_str}.csv', sep=';') 
    df_trades = pd.read_csv(f'dataset/trades_round_2_day_{day_str}.csv', sep=';')

    # Filter datasets for the chosen product
    p_df = df_prices[df_prices['product'] == product].copy()
    t_df = df_trades[df_trades['symbol'] == product].copy()

    # Filter out invalid ticks where the order book clears out
    p_df = p_df[p_df['mid_price'] > 0]

    # 2. Calculate EMAs directly on the mid_price
    p_df['EMA_5'] = p_df['mid_price'].ewm(span=5, adjust=False).mean()
    p_df['EMA_20'] = p_df['mid_price'].ewm(span=20, adjust=False).mean()
    p_df['EMA_50'] = p_df['mid_price'].ewm(span=50, adjust=False).mean()

    fig = go.Figure()

    # -------------------------------------------------------------
    # 3. ADD ORDER BOOK DEPTH (Plotted first to stay in background)
    # -------------------------------------------------------------
    
    # BIDS - Fading blue lines for depth
    bid_colors = ['rgba(50, 100, 250, 1)', 'rgba(50, 100, 250, 0.75)', 'rgba(50, 100, 250, 0.5)']
    for i in range(1, 4):
        col_name = f'bid_price_{i}'
        if col_name in p_df.columns:
            fig.add_trace(go.Scatter(
                x=p_df['timestamp'],
                y=p_df[col_name],
                mode='lines',
                line=dict(color=bid_colors[i-1], width=1 if i==1 else 0.5),
                name=f'Bid L{i}',
                hoverinfo='skip' 
            ))

    # ASKS - Fading red lines for depth
    ask_colors = ['rgba(250, 50, 50, 1)', 'rgba(250, 50, 50, 0.75)', 'rgba(250, 50, 50, 0.5)']
    for i in range(1, 4):
        col_name = f'ask_price_{i}'
        if col_name in p_df.columns:
            fig.add_trace(go.Scatter(
                x=p_df['timestamp'],
                y=p_df[col_name],
                mode='lines',
                line=dict(color=ask_colors[i-1], width=1 if i==1 else 0.5),
                name=f'Ask L{i}',
                hoverinfo='skip'
            ))

    # -------------------------------------------------------------
    # 4. ADD FOREGROUND ELEMENTS (Mid Price Line, EMAs, & Trades)
    # -------------------------------------------------------------
    
    # Add Mid Price Line
    fig.add_trace(go.Scatter(
        x=p_df['timestamp'],
        y=p_df['mid_price'],
        mode='lines',
        line=dict(color='#ffffff', width=2), # White line for main price
        name='Mid Price'
    ))

    # Add EMAs
    fig.add_trace(go.Scatter(
        x=p_df['timestamp'], y=p_df['EMA_5'], mode='lines',
        line=dict(color='#00e676', width=1.5), name='EMA 5' # Green
    ))
    
    fig.add_trace(go.Scatter(
        x=p_df['timestamp'], y=p_df['EMA_20'], mode='lines',
        line=dict(color='#ffea00', width=1.5), name='EMA 20' # Yellow
    ))
    
    fig.add_trace(go.Scatter(
        x=p_df['timestamp'], y=p_df['EMA_50'], mode='lines',
        line=dict(color='#d500f9', width=1.5), name='EMA 50' # Purple
    ))

    # Add Trade Markers
    if not t_df.empty:
        fig.add_trace(go.Scatter(
            x=t_df['timestamp'],
            y=t_df['price'],
            mode='markers',
            marker=dict(color='#ff9800', size=8, symbol='circle', line=dict(color='white', width=1)), # Orange dots
            name='Trades',
            customdata=t_df['quantity'],
            hovertemplate='Price: %{y}<br>Volume: %{customdata}<extra></extra>'
        ))

    # 5. Format Layout
    fig.update_layout(
        title=f'{product} - Day {day_str} (Mid Price & EMAs) w/ Order Depth',
        xaxis_title='Timestamp',
        yaxis_title='Price (XIRECs)',
        template='plotly_dark', 
        xaxis_rangeslider_visible=True,
        yaxis=dict(fixedrange=False), 
        hovermode='x unified', 
        height=700
    )

    # Display the interactive chart in the browser
    fig.show(renderer="browser")

# --- RUN THE VISUALIZATIONS ---
for prd in ["ASH_COATED_OSMIUM", "INTARIAN_PEPPER_ROOT"]:
    for i in ["-1", "0", "1"]:
        plot_prosperity_day(i, prd)